In [0]:
# =============================================================================
# NOTEBOOK  : silver_purchase_orders
# PURPOSE   : bronze.purchase_orders → silver.purchase_orders
# LAYER     : Silver (clean, flatten nested JSON, CDC merge)
# FREQUENCY : Daily + CDC (availableNow)
# TRIGGER   : availableNow
#
# TRANSFORM:
#   - order_date / expected_delivery_date / actual_delivery_date : String → DateType
#   - unit_cost        : Double → Decimal(10,2)
#   - quality_rating   : Double → Decimal(4,2)
#   - delivery_notes   : nested JSON string → flattened into 4 columns:
#                        carrier, tracking_number, delivery_status, delivery_notes_text
#   - order_year_month : derived from order_date → partition column
#
# MERGE & DEDUP LOGIC:
#   Dedup     : window on po_id ordered by ingested_at DESC
#               keep most recently ingested version of same po_id
#               CDC source may send same PO multiple times in same file
#               ingested_at used BEFORE .select() so it's still available
#
#   Merge key : po_id
#   Case 1    : po_id NOT in silver → INSERT
#   Case 2    : po_id IN silver, CDC fields changed → UPDATE
#               Updatable: status
#               All other fields immutable once PO is created
#   Case 3    : po_id IN silver, no change → IGNORE
# =============================================================================

# ── CELL 1: Imports & Config ──────────────────────────────────────────────────
import sys, json
sys.path.insert(0, "/Workspace/Shared/mini_project_a3t7")

from utils.audit import start_audit, end_audit
from utils.schema_registry import SILVER_PURCHASE_ORDERS_SCHEMA

from pyspark.sql.functions import (
    current_timestamp, lit, col,
    to_date, get_json_object, when,
    date_format, row_number
)
from pyspark.sql.types import DecimalType
from pyspark.sql.window import Window
from delta.tables import DeltaTable

with open("/Workspace/Shared/mini_project_a3t7/config/config.json") as f:
    cfg = json.load(f)

SOURCE_TABLE = cfg["tables"]["bronze_purchase_orders"]
TARGET_TABLE = cfg["tables"]["silver_purchase_orders"]
CHECKPOINT   = cfg["checkpoint_paths"]["silver_purchase_orders"]
PIPELINE     = "silver_purchase_orders"

In [0]:
# ── foreachBatch function + Stream ────────────────────────────────────

def process_microbatch(micro_batch_df, microbatch_id):
    """
    Called by Spark for every micro-batch of bronze purchase order rows.

    DEDUP  : window on po_id ordered by ingested_at DESC
             done BEFORE .select() so ingested_at is still available
             keeps most recently ingested version of same po_id

    MERGE  :
      Case 1 → po_id not in silver                    → INSERT
      Case 2 → po_id in silver, CDC fields changed    → UPDATE
               only: status
               all other columns immutable once PO is created
      Case 3 → po_id in silver, no change             → IGNORE
    """

    import sys
    sys.path.insert(0, "/Workspace/Shared/mini_project_a3t7")
    from utils.audit import start_audit, end_audit
    from utils.schema_registry import SILVER_PURCHASE_ORDERS_SCHEMA

    if micro_batch_df.isEmpty():
        return

    run_id = start_audit(spark, PIPELINE, TARGET_TABLE, SOURCE_TABLE)

    try:
        rows_read = micro_batch_df.count()

        # ── TRANSFORM ─────────────────────────────────────────────────────────
        df = (
            micro_batch_df

            # 1. Date columns: String → DateType
            .withColumn("order_date",
                to_date(col("order_date")))
            .withColumn("expected_delivery_date",
                to_date(col("expected_delivery_date")))
            .withColumn("actual_delivery_date",
                to_date(col("actual_delivery_date")))

            # 2. Numeric precision
            .withColumn("unit_cost",
                col("unit_cost").cast(DecimalType(10, 2)))
            .withColumn("quality_rating",
                col("quality_rating").cast(DecimalType(4, 2)))

            # 3. Flatten delivery_notes nested JSON string
            #    Source: {"carrier":"UPS","tracking_number":"TRACK831632",
            #             "delivery_status":"delayed","notes":"Address issue"}
            .withColumn("carrier",
                get_json_object(col("delivery_notes"), "$.carrier"))
            .withColumn("tracking_number",
                get_json_object(col("delivery_notes"), "$.tracking_number"))
            .withColumn("delivery_status",
                get_json_object(col("delivery_notes"), "$.delivery_status"))
            .withColumn("delivery_notes_text",
                get_json_object(col("delivery_notes"), "$.notes"))

            # 4. Derive partition column from order_date
            #    format: "2023-01" — monthly granularity
            .withColumn("order_year_month",
                date_format(col("order_date"), "yyyy-MM"))

            # 5. Silver audit columns
            #    ingested_at carried from bronze — tracks when raw data landed
            #    processed_at set here — tracks when silver processed it
            #    gap = pipeline lag between bronze ingestion and silver processing
            .withColumn("processed_at",    current_timestamp())
            .withColumn("pipeline_run_id", lit(run_id))
            .withColumn("source_system",   lit("purchase_orders_csv"))
            # ingested_at already in bronze df — carried forward automatically
            # by .select() since it is in SILVER_PURCHASE_ORDERS_SCHEMA
        )

        # ── DEDUP WITHIN MICRO-BATCH ──────────────────────────────────────────
        # Done BEFORE .select() so ingested_at is still available as tiebreaker
        # CDC source may send same po_id multiple times — keep most recent
        # ingested_at tells us which copy arrived latest in bronze
        # Status lifecycle priority — higher = more recent state
        df = df.dropDuplicates(["po_id", "status"])
        
        df = df.withColumn(
            "status_priority",
            when(col("status") == "pending",   lit(1))
            .when(col("status") == "shipped",  lit(2))
            .when(col("status") == "delivered",lit(3))
            .when(col("status") == "cancelled",lit(3))
            .otherwise(lit(0))
        )

        window = (
            Window
            .partitionBy("po_id")
            .orderBy(col("status_priority").desc())
        )

        df = (
            df
            .withColumn("_row_num", row_number().over(window))
            .filter(col("_row_num") == 1)
            .drop("_row_num", "status_priority")
        )

        rows_after_dedup = df.count()
        if rows_read != rows_after_dedup:
            print(f"[DEDUP] dropped {rows_read - rows_after_dedup} "
                  f"duplicate PO rows in microbatch_id={microbatch_id}")

        # ── ENFORCE SILVER SCHEMA ─────────────────────────────────────────────
        # Done AFTER dedup so ingested_at was available for window ordering
        # Drops all bronze-only columns:
        # (delivery_notes raw string, source_file, ingested_date, _source etc.)
        df = df.select([f.name for f in SILVER_PURCHASE_ORDERS_SCHEMA.fields])

        # ── MERGE INTO SILVER ──────────────────────────────────────────────────
        # Case 2: po_id matched, CDC fields changed → UPDATE 3 fields only
        # Case 1: po_id not matched                 → INSERT
        # Case 3: po_id matched, no change          → no rule fires → IGNORE
        (
            DeltaTable.forName(spark, TARGET_TABLE).alias("t")
            .merge(
                df.alias("s"),
                "t.po_id = s.po_id"
            )
            .whenMatchedUpdate(
                condition="t.status != s.status",
                set={
                    "status":          "s.status",
                    "processed_at":    "current_timestamp()",
                    "pipeline_run_id": f"'{run_id}'"
                }
            )
            .whenNotMatchedInsertAll()
            .execute()
        )

        metrics = (
            spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE} LIMIT 1")
            .select("operationMetrics")
            .collect()[0][0]
        )
        rows_inserted = int(metrics.get("numTargetRowsInserted", 0))
        rows_updated  = int(metrics.get("numTargetRowsUpdated", 0))

        end_audit(
            spark, run_id, "SUCCESS",
            rows_read=rows_read,
            rows_written=rows_inserted + rows_updated,
            extra={
                "rows_inserted": str(rows_inserted),
                "rows_updated":  str(rows_updated),
                "microbatch_id": str(microbatch_id)
            }
        )

    except Exception as e:
        end_audit(spark, run_id, "FAILED", error=str(e))
        raise


# ── READ BRONZE AS STREAM ─────────────────────────────────────────────────────
bronze_stream = (
    spark.readStream
    .format("delta")
    .option("ignoreChanges", "true")
    .table(SOURCE_TABLE)
)

# ── WRITE WITH foreachBatch ───────────────────────────────────────────────────
query = (
    bronze_stream.writeStream
    .foreachBatch(process_microbatch)
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

In [0]:
# ── Verify last run ───────────────────────────────────────────────────
from pyspark.sql.functions import col

# 1. Audit status
spark.read.table("azure3_team7_project.platform.pipeline_audit") \
    .filter(col("pipeline_name") == PIPELINE) \
    .orderBy(col("start_time").desc()) \
    .limit(1) \
    .select("status", "rows_read", "rows_written", "extra_metadata") \
    .show(truncate=False)

# 2. Silver row count
print("Silver row count:", spark.read.table(TARGET_TABLE).count())

# 3. Nulls in key columns
print("Null key columns:",
    spark.read.table(TARGET_TABLE)
    .filter(col("po_id").isNull())
    .count()
)

# 4. Delta history
spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE} LIMIT 1") \
    .select("operation", "operationMetrics") \
    .show(truncate=False)